# Conversion Prediction Model

# Objective
**Problem:** Predict which free-tier (non-customer) companies are likely to convert to paying customers_df_df_df_df_df_df_df_df_df within the next 30 days.

**Output:** Weekly prioritized list (generated on Sundays).

# Assumptions
+ Alexa rank Null means that the company has no webpage, thus we assign 16000001 to these cases

# Table of Contents

In [58]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, IFrame
from ydata_profiling import ProfileReport
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

# Load Data
- Load data
- Preprocess and standardize columns
- Check data integrity and basic statistics

In [59]:

# Load data
customers_df = pd.read_csv("../data/customers.csv")
noncustomers_df = pd.read_csv("../data/noncustomers.csv")
usage_actions_df = pd.read_csv("../data/usage_actions.csv")

# Parse dates
customers_df['CLOSEDATE'] = pd.to_datetime(customers_df['CLOSEDATE'])
usage_actions_df['WHEN_TIMESTAMP'] = pd.to_datetime(usage_actions_df['WHEN_TIMESTAMP'])

# Sort by CLOSEDATE and WHEN_TIMESTAMP
customers_df = customers_df.sort_values(by=['CLOSEDATE'])
usage_actions_df = usage_actions_df.sort_values(by=['WHEN_TIMESTAMP'])

# All columns to upper case
customers_df.columns = customers_df.columns.str.upper()
noncustomers_df.columns = noncustomers_df.columns.str.upper()
usage_actions_df.columns = usage_actions_df.columns.str.upper()


# Check data
print(f"customers:     {customers_df.shape}  — IDs {customers_df['ID'].min()}–{customers_df['ID'].max()}")
print(f"noncustomers:  {noncustomers_df.shape}  — IDs {noncustomers_df['ID'].min()}–{noncustomers_df['ID'].max()}")
print(f"usage_actions: {usage_actions_df.shape}")
print(f"Usage date range: {usage_actions_df['WHEN_TIMESTAMP'].min().date()} → {usage_actions_df['WHEN_TIMESTAMP'].max().date()}")
print(f"Unique ids: {usage_actions_df['ID'].nunique()}")

cust_ids   = set(customers_df["ID"])
noncust_ids= set(noncustomers_df["ID"])
print(f"ID overlap (customers ∩ noncustomers): {len(cust_ids & noncust_ids)}")  

REF_DATE = usage_actions_df["WHEN_TIMESTAMP"].max()
print(f"Reference date ('today'): {REF_DATE.date()}")


customers:     (200, 6)  — IDs 1–200
noncustomers:  (5003, 4)  — IDs 201–5200
usage_actions: (25387, 10)
Usage date range: 2019-01-07 → 2020-07-27
Unique ids: 3569
ID overlap (customers ∩ noncustomers): 0
Reference date ('today'): 2020-07-27


# EDA
+ Missing values
+ Using ydata_profiling for detailed and shareable insights found in ./reports

In [60]:
def missing_summary(df, name):
    miss = df.isna().sum()
    miss = miss[miss > 0]

    print(f"\n{name} columns with missing values:")
    if miss.empty:
        print("None")
        return

    out = pd.DataFrame({
        "column": miss.index,
        "missing_count": miss.values,
        "missing_pct": (miss.values / len(df) * 100).round(2),
    }).sort_values(["missing_pct", "missing_count"], ascending=False)

    print(out.to_string(index=False))


missing_summary(customers_df, "CUSTOMERS")
missing_summary(noncustomers_df, "NON-CUSTOMERS")
missing_summary(usage_actions_df, "USAGE")



CUSTOMERS columns with missing values:
        column  missing_count  missing_pct
      INDUSTRY            129         64.5
EMPLOYEE_RANGE              2          1.0

NON-CUSTOMERS columns with missing values:
        column  missing_count  missing_pct
      INDUSTRY           3725        74.46
EMPLOYEE_RANGE            532        10.63
    ALEXA_RANK            114         2.28

USAGE columns with missing values:
None


In [61]:
# Company-level comparison: customers vs non-customers
historical_customers_df = customers_df[customers_df["CLOSEDATE"] <= REF_DATE].copy()

reports_dir = Path("../reports")
reports_dir.mkdir(parents=True, exist_ok=True)

customer_profile    = ProfileReport(historical_customers_df, title="Customers",     minimal=True)
noncustomer_profile = ProfileReport(noncustomers_df,         title="Non-customers", minimal=True)

customer_profile.compare(noncustomer_profile).to_file(reports_dir / "company_comparison.html")

display(IFrame(src=str(reports_dir / "company_comparison.html"), width="100%", height=600))

c:\Users\totic\anaconda3\Lib\site-packages\ydata_profiling\compare_reports.py:191: UserWarning: The datasets being profiled have a different set of columns. Only the left side profile will be calculated.
  warnings.warn(


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 6/6 [00:00<00:00, 1200.83it/s]


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:00<00:00, 227.13it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

# Feature Engineering
+ Assumption: Alexa rank Null means that the company has no webpage, thus we assign 16000001 to these cases
+ INDUSTRY nulls could be reduced by doing Web Scraping or buying a company industry database
+ Employee range to the middle of the range
+ One Hot Encoding of INDUSTRY
+ Build date features

In [62]:
# Merge customers and noncustomers into one aligned company table
# Keep full customers schema and add LABEL (1=customer, 0=noncustomer)
customers_base = customers_df.copy()
noncustomers_base = noncustomers_df.copy()

customers_base["LABEL"] = 1
noncustomers_base["LABEL"] = 0

# Use customers schema as canonical so CLOSEDATE/MRR are preserved in companies_df
target_cols = list(customers_base.columns)
noncustomers_base = noncustomers_base.reindex(columns=target_cols)

# Align dtypes before concat to avoid pandas warnings and casting errors
for col, dtype in customers_base[target_cols].dtypes.items():
    if pd.api.types.is_datetime64_any_dtype(dtype):
        noncustomers_base[col] = pd.to_datetime(noncustomers_base[col], errors="coerce")
    elif pd.api.types.is_integer_dtype(dtype):
        # Integer columns may contain NA for noncustomers (e.g., customer-only fields)
        # so use pandas nullable integer dtype.
        noncustomers_base[col] = pd.to_numeric(noncustomers_base[col], errors="coerce").astype("Int64")
        customers_base[col] = customers_base[col].astype("Int64")
    elif pd.api.types.is_float_dtype(dtype):
        noncustomers_base[col] = pd.to_numeric(noncustomers_base[col], errors="coerce").astype(dtype)
    else:
        noncustomers_base[col] = noncustomers_base[col].astype(dtype)

companies_df = pd.concat([customers_base[target_cols], noncustomers_base], axis=0, ignore_index=True)

print("Merged companies_df shape:", companies_df.shape)
print("Columns:", companies_df.columns.tolist())
print("Label split:")
print(companies_df["LABEL"].value_counts().sort_index())


IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer

In [ ]:
companies_df

,CLOSEDATE,MRR,ALEXA_RANK,EMPLOYEE_RANGE,INDUSTRY,ID,EMPLOYEE_RANGE_NUM,LABEL
0,2019-01-15,300.96,16000001,6 to 10,OTHER,42,8.0,1
1,2019-01-17,49.07,10390859,2 to 5,OTHER,20,3.0,1
2,2019-01-27,209.98,273063,26 to 50,Technology - Software,174,38.0,1
3,2019-01-30,40.00,2610402,2 to 5,RENEWABLES_ENVIRONMENT,1,3.0,1
4,2019-01-30,400.00,16000001,11 to 25,OTHER,25,18.0,1
...,...,...,...,...,...,...,...,...
5198,NaT,NaN,16000001,NaN,NaN,637,NaN,0
5199,NaT,NaN,20183,1001 to 10000,Non-Profit/Educational Institution,4921,5500.0,0
5200,NaT,NaN,16000001,6 to 10,NaN,1215,8.0,0
5201,NaT,NaN,16000001,11 to 25,NaN,2693,18.0,0


In [ ]:
# For null ALEXA_RANK assume they where not found and page-rank will be weak
max_rank = companies_df['ALEXA_RANK'].max()
companies_df['ALEXA_RANK'] = np.where(companies_df['ALEXA_RANK'].isnull(), max_rank, companies_df['ALEXA_RANK'])

In [ ]:
# Employee range to numberic
# For a better comparison, map EMPLOYEE_RANGE with explicit dictionaries.
EMPLOYEE_RANGE_TO_MID = {
    "1": 1.0,
    "2 to 5": 3.0,
    "6 to 10": 8.0,
    "11 to 25": 18.0,
    "26 to 50": 38.0,
    "51 to 200": 125.0,
    "201 to 1000": 600.0,
    "1001 to 10000": 5500.0,
    "10,001 or more": 10001.0,
}

companies_df["EMPLOYEE_RANGE"] = companies_df["EMPLOYEE_RANGE"].map(EMPLOYEE_RANGE_TO_MID)


In [ ]:


def encode_industries(df, column='INDUSTRY', max_cats=10):
    """
    Standardizes industry names, maps similar categories, 
    and performs One-Hot Encoding with a 'Long Tail' capture.
    """
    # 1. Basic Cleaning
    series = df[column].fillna('UNKNOWN').astype(str).str.upper().str.strip()
    
    # 2. Strategic Mapping (Grouping synonyms)
    # Using regex=True in replace allows us to catch substrings more easily
    mapping = {
        r'.*SOFTWARE.*|.*SAAS.*|.*TECHNOL.*': 'TECH_SOFTWARE',
        r'.*IT_SERVICES.*|.*INFORMATION_TECHNOLOGY.*|.*COMPUTER_NETWORK.*': 'TECH_IT',
        r'.*CONSULTING.*|.*ADVISORY.*': 'CONSULTING',
        r'.*STAFFING.*|.*RECRUITING.*': 'HR_STAFFING',
        r'.*FINANCE.*|.*FINANCIAL.*|.*BANKING.*|.*ACCOUNTING.*|.*INSURANCE.*': 'FINANCE',
        r'.*ENERGY.*|.*RENEWABLES.*|.*UTILITIES.*': 'ENERGY_UTILITIES',
        r'.*EDUCATION.*|.*E_LEARNING.*': 'EDUCATION'
    }
    
    # Apply mapping using regex to catch messy variations
    clean_series = series.replace(mapping, regex=True)
    
    # 3. One-Hot Encoding
    # max_categories groups the smallest categories into an 'infrequent' column
    encoder = OneHotEncoder(
        max_categories=max_cats,
        handle_unknown='infrequent_if_exist',
        sparse_output=False
    )
    
    # Fit and transform
    encoded_array = encoder.fit_transform(clean_series.to_frame())
    
    # Get clean feature names (e.g., IND_TECH_SOFTWARE)
    feature_names = encoder.get_feature_names_out([column])
    
    # Create output DataFrame
    ohe_df = pd.DataFrame(
        encoded_array, 
        columns=feature_names, 
        index=df.index
    ).astype(int)
    
    return pd.concat([df, ohe_df], axis=1)

# Usage:
companies_df = encode_industries(companies_df, 'INDUSTRY', max_cats=8)
companies_df

array(['OTHER', 'Technology - Software', 'RENEWABLES_ENVIRONMENT',
       'ACCOUNTING', 'COMPUTER_SOFTWARE', 'Business Consulting',
       'OIL_ENERGY', 'BUILDING_MATERIALS',
       'INFORMATION_TECHNOLOGY_AND_SERVICES', 'MARKETING_AND_ADVERTISING',
       'COMPUTER_NETWORK_SECURITY', 'Consulting/Advisory', 'SaaS',
       'REAL_ESTATE', 'MECHANICAL_OR_INDUSTRIAL_ENGINEERING', 'Other',
       'MANAGEMENT_CONSULTING', 'E_LEARNING', 'FINANCIAL_SERVICES',
       'INDUSTRIAL_AUTOMATION', 'CONSUMER_SERVICES', 'Software',
       'EDUCATION_MANAGEMENT', 'NON_PROFIT_ORGANIZATION_MANAGEMENT',
       'MEDICAL_DEVICES', 'HIGHER_EDUCATION', 'FOOD_BEVERAGES',
       'Recruiting/Staffing', 'Computers, Networking, Software, Technol',
       'RETAIL', 'HEALTH_WELLNESS_AND_FITNESS',
       'Banking/Financial Services', 'UTILITIES',
       'ENVIRONMENTAL_SERVICES', 'INSURANCE', 'STAFFING_AND_RECRUITING',
       'DESIGN', 'WINE_AND_SPIRITS', 'Cannabinoid',
       'LEISURE_TRAVEL_TOURISM'], dtype=object)

In [ ]:
# Group small categories into 'OTHER'
top_inds = customers_df['INDUSTRY'].unique()
customers_df['INDUSTRY'] = customers_df['INDUSTRY'].apply(lambda x: x if x in top_inds else 'OTHER')

# Fill NaNs specifically to ensure they get a column
customers_df['INDUSTRY'] = customers_df['INDUSTRY'].fillna('UNKNOWN')

# One-hot encode everything in one shot
customers_df_a = pd.get_dummies(customers_df, columns=['INDUSTRY'], prefix='IND')
customers_df_a

,CLOSEDATE,MRR,ALEXA_RANK,EMPLOYEE_RANGE,ID,EMPLOYEE_RANGE_NUM,IND_ACCOUNTING,IND_BUILDING_MATERIALS,IND_Banking/Financial Services,IND_Business Consulting,...,IND_REAL_ESTATE,IND_RENEWABLES_ENVIRONMENT,IND_RETAIL,IND_Recruiting/Staffing,IND_STAFFING_AND_RECRUITING,IND_SaaS,IND_Software,IND_Technology - Software,IND_UTILITIES,IND_WINE_AND_SPIRITS
58,2019-01-15,300.96,16000001,6 to 10,42,8.0,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
67,2019-01-17,49.07,10390859,2 to 5,20,3.0,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,2019-01-27,209.98,273063,26 to 50,174,38.0,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
41,2019-01-30,40.00,2610402,2 to 5,1,3.0,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False
18,2019-01-30,400.00,16000001,11 to 25,25,18.0,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,2020-08-05,400.00,3317099,11 to 25,8,18.0,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
167,2020-08-12,200.00,483736,6 to 10,27,8.0,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
129,2020-08-14,800.00,16000001,6 to 10,35,8.0,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
180,2020-08-15,250.00,6426139,11 to 25,185,18.0,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [44]:
usage_actions_df

,WHEN_TIMESTAMP,ACTIONS_CRM_CONTACTS,ACTIONS_CRM_COMPANIES,ACTIONS_CRM_DEALS,ACTIONS_EMAIL,USERS_CRM_CONTACTS,USERS_CRM_COMPANIES,USERS_CRM_DEALS,USERS_EMAIL,ID
2287,2019-01-07,0,0,0,0,0,0,0,0,4916
2346,2019-01-07,0,0,0,0,0,0,0,0,4432
12713,2019-01-07,136,0,1,0,11,0,1,0,157
12827,2019-01-07,1,1,19,0,1,1,1,0,162
13124,2019-01-07,1,0,0,0,1,0,0,0,4412
...,...,...,...,...,...,...,...,...,...,...
7000,2020-07-27,121,343,94,0,9,14,11,0,51
12384,2020-07-27,1,0,0,0,1,0,0,0,3041
20639,2020-07-27,166,0,106,1,7,0,6,1,32
12307,2020-07-27,22,0,3,0,2,0,1,0,4834


In [45]:

print("\n[3] Feature Engineering...")

ACTION_COLS = ["ACTIONS_CRM_CONTACTS","ACTIONS_CRM_COMPANIES","ACTIONS_CRM_DEALS","ACTIONS_EMAIL"]
USER_COLS   = ["USERS_CRM_CONTACTS","USERS_CRM_COMPANIES","USERS_CRM_DEALS","USERS_EMAIL"]
METRIC_COLS = ACTION_COLS + USER_COLS
WINDOWS     = {"7d": 7, "14d": 14, "30d": 30, "60d": 60}


def build_usage_features(portal_ids, cutoff_dates_map=None):
    """
    Build time-windowed usage features per portal ID.

    Parameters
    ----------
    portal_ids : iterable
        Portal IDs to compute features for.
    cutoff_dates_map : dict, optional
        Mapping {ID: cutoff_timestamp}. If not provided, REF_DATE is used for all IDs.
        This prevents leakage for customers by excluding post-conversion usage.
    """
    portal_ids = list(portal_ids)
    cutoffs = cutoff_dates_map or {}

    # Work only with needed rows/columns for speed and consistent numeric ops.
    cols_needed = ["ID", "WHEN_TIMESTAMP"] + ACTION_COLS + USER_COLS
    relevant_usage = usage_actions_df.loc[
        usage_actions_df["ID"].isin(portal_ids), cols_needed
    ].copy()
    relevant_usage["WHEN_TIMESTAMP"] = pd.to_datetime(relevant_usage["WHEN_TIMESTAMP"])
    relevant_usage = relevant_usage.sort_values(["ID", "WHEN_TIMESTAMP"])

    all_rows = []

    for pid in portal_ids:
        cutoff = pd.Timestamp(cutoffs.get(pid, REF_DATE))
        portal_usage = relevant_usage.loc[
            (relevant_usage["ID"] == pid) & (relevant_usage["WHEN_TIMESTAMP"] < cutoff)
        ]

        row = {"ID": pid}

        # Window-based features
        for wname, wdays in WINDOWS.items():
            win_start = cutoff - pd.Timedelta(days=wdays)
            w_data = portal_usage.loc[portal_usage["WHEN_TIMESTAMP"] >= win_start]

            for col in METRIC_COLS:
                row[f"{col}_sum_{wname}"] = float(w_data[col].sum())
                row[f"{col}_mean_{wname}"] = float(w_data[col].mean()) if not w_data.empty else 0.0

            row[f"active_days_{wname}"] = int(w_data["WHEN_TIMESTAMP"].nunique())
            row[f"total_actions_{wname}"] = float(w_data[ACTION_COLS].sum().sum())
            row[f"total_users_{wname}"] = float(w_data[USER_COLS].sum().sum())

        # Lifetime features
        row["total_actions_lifetime"] = float(portal_usage[ACTION_COLS].sum().sum())
        row["total_users_lifetime"] = float(portal_usage[USER_COLS].sum().sum())
        row["active_days_lifetime"] = int(portal_usage["WHEN_TIMESTAMP"].nunique())

        if not portal_usage.empty:
            first_usage = portal_usage["WHEN_TIMESTAMP"].min()
            last_usage = portal_usage["WHEN_TIMESTAMP"].max()

            row["days_since_first_usage"] = int((cutoff - first_usage).days)
            row["days_since_last_usage"] = int((cutoff - last_usage).days)
            row["usage_recency_score"] = 1.0 / (row["days_since_last_usage"] + 1)

            # Trend: last 30 days vs previous 30 days
            last30 = portal_usage.loc[
                portal_usage["WHEN_TIMESTAMP"] >= cutoff - pd.Timedelta(days=30), ACTION_COLS
            ].sum().sum()
            prev30 = portal_usage.loc[
                (portal_usage["WHEN_TIMESTAMP"] >= cutoff - pd.Timedelta(days=60))
                & (portal_usage["WHEN_TIMESTAMP"] < cutoff - pd.Timedelta(days=30)),
                ACTION_COLS,
            ].sum().sum()
            row["action_trend_ratio"] = float(last30 / (prev30 + 1.0))

            # Distinct modules used at least once
            row["module_diversity"] = int((portal_usage[ACTION_COLS].sum() > 0).sum())
        else:
            row["days_since_first_usage"] = 999
            row["days_since_last_usage"] = 999
            row["usage_recency_score"] = 0.0
            row["action_trend_ratio"] = 0.0
            row["module_diversity"] = 0

        all_rows.append(row)

    return pd.DataFrame(all_rows).set_index("ID")


def build_company_features(df):
    """Static company features from customers or noncustomers dataframe."""
    out = df.set_index("ID").copy()
    
    # Alexa rank: sentinel → missing
    out["ALEXA_RANK_CLEAN"] = out["ALEXA_RANK"].replace(16000001, np.nan)
    out["IS_UNRANKED"]      = (out["ALEXA_RANK"] == 16000001).astype(int)
    out["LOG_ALEXA_RANK"]   = np.log10(out["ALEXA_RANK_CLEAN"] + 1)
    
    # Employee range → ordinal
    emp_map = {"1 to 5": 1, "6 to 10": 2, "11 to 25": 3, "26 to 50": 4,
               "51 to 200": 5, "201 to 1000": 6, "1001 to 10000": 7, "10001+": 8}
    out["EMP_ORDINAL"] = out["EMPLOYEE_RANGE"].map(emp_map)
    
    # Industry one-hot (top categories)
    top_inds = ["COMPUTER_SOFTWARE","MARKETING_AND_ADVERTISING","IT_SERVICES",
                "INTERNET","FINANCIAL_SERVICES","HIGHER_EDUCATION","CONSUMER_SERVICES"]
    for ind in top_inds:
        out[f"IND_{ind}"] = (out["INDUSTRY"] == ind).astype(int)
    out["IND_OTHER"]    = (~out["INDUSTRY"].isin(top_inds) & out["INDUSTRY"].notna()).astype(int)
    out["IND_UNKNOWN"]  = out["INDUSTRY"].isna().astype(int)
    
    return out


# Build features for CUSTOMERS (with leakage prevention: use pre-close usage only)
print("  Building customer features (with data leakage prevention)...")
cust_cutoffs = {row["ID"]: row["CLOSEDATE"] for _, row in customers_df.iterrows()}
cust_usage_feats = build_usage_features(list(cust_ids), cust_cutoffs)
cust_company_feats = build_company_features(customers_df.drop(columns=["CLOSEDATE","MRR"], errors="ignore"))

cust_feats = cust_company_feats.join(cust_usage_feats, how="left")
cust_feats["LABEL"] = 1

# Build features for NON-CUSTOMERS (usage up to REF_DATE)
print("  Building non-customer features...")
nc_usage_feats = build_usage_features(list(noncust_ids))
nc_company_feats = build_company_features(noncustomers_df)

nc_feats = nc_company_feats.join(nc_usage_feats, how="left")
nc_feats["LABEL"] = 0

print(f"  Customer features: {cust_feats.shape}")
print(f"  Non-customer features: {nc_feats.shape}")


[3] Feature Engineering...
  Building customer features (with data leakage prevention)...
  Building non-customer features...
  Customer features: (200, 102)
  Non-customer features: (5003, 102)


In [48]:
cust_usage_feats

,ACTIONS_CRM_CONTACTS_sum_7d,ACTIONS_CRM_CONTACTS_mean_7d,ACTIONS_CRM_COMPANIES_sum_7d,ACTIONS_CRM_COMPANIES_mean_7d,ACTIONS_CRM_DEALS_sum_7d,ACTIONS_CRM_DEALS_mean_7d,ACTIONS_EMAIL_sum_7d,ACTIONS_EMAIL_mean_7d,USERS_CRM_CONTACTS_sum_7d,USERS_CRM_CONTACTS_mean_7d,...,total_actions_60d,total_users_60d,total_actions_lifetime,total_users_lifetime,active_days_lifetime,days_since_first_usage,days_since_last_usage,usage_recency_score,action_trend_ratio,module_diversity
ID,,,,,,,,,,,,,,,,,,,,,
1,208.0,208.0,61.0,61.0,87.0,87.0,7.0,7.0,4.0,4.0,...,917.0,30.0,917.0,30.0,4,23,2,0.333333,917.000000,4
2,2.0,2.0,0.0,0.0,14.0,14.0,0.0,0.0,1.0,1.0,...,123.0,12.0,920.0,59.0,14,106,1,0.500000,0.333333,4
3,1407.0,1407.0,220.0,220.0,439.0,439.0,5.0,5.0,24.0,24.0,...,12461.0,399.0,35455.0,1213.0,31,218,1,0.500000,1.816271,4
4,1510.0,1510.0,2.0,2.0,38.0,38.0,0.0,0.0,11.0,11.0,...,7528.0,116.0,39917.0,557.0,53,406,7,0.125000,3.468249,3
5,17.0,17.0,18.0,18.0,27.0,27.0,0.0,0.0,4.0,4.0,...,806.0,94.0,1391.0,213.0,30,255,3,0.250000,1.123684,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
196,4.0,4.0,1.0,1.0,81.0,81.0,4.0,4.0,1.0,1.0,...,183.0,16.0,317.0,35.0,19,190,1,0.500000,183.000000,4
197,3.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,22.0,10.0,119.0,35.0,43,331,2,0.333333,1.875000,3
198,452.0,452.0,20.0,20.0,25.0,25.0,0.0,0.0,3.0,3.0,...,739.0,18.0,739.0,18.0,3,22,1,0.500000,739.000000,3
